# Notebook 01 — Generate and Ingest Synthetic Banking Data
## Contoso Banque — Churn Analysis Workshop

> **All data generated in this notebook is entirely synthetic and fictional.**
> It does not represent any real bank, customer, or financial institution.
> "Contoso Banque" is a fictional entity used for educational purposes only.

---

## What This Notebook Does

This notebook generates five synthetic datasets representing a fictional European retail bank:

| Dataset | Rows | Description |
|---|---|---|
| `customers` | 10,000 | Demographics, tenure, risk profile, digital activity, churn label |
| `accounts` | ~15,000 | Bank accounts: balances, account type, trends |
| `products` | 10 | Product catalogue: savings, credit, insurance, etc. |
| `customer_products` | ~30,000 | Which products each customer holds |
| `transactions` | ~175,000 | Individual transactions: amounts, channels, categories |

Data is written to:
- `Files/churn/raw/` (raw staging in Parquet format)
- `Tables/` (Delta tables — queryable by SQL, Power BI, Data Agent)

## Prerequisites
- This notebook must be run with `ChurnAnalysisLH` lakehouse attached.
- No external libraries required — uses PySpark only.
- Estimated run time: 3–8 minutes depending on capacity SKU.

## Cell 1 — Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, rand, round as spark_round
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)
import random
import math
from datetime import date, timedelta

spark = SparkSession.builder.getOrCreate()

# Configuration
RANDOM_SEED = 42
NUM_CUSTOMERS = 10_000

random.seed(RANDOM_SEED)

print("✅ Spark session ready")
print(f"   Spark version: {spark.version}")
print(f"   Random seed: {RANDOM_SEED}")
print(f"   Target customers: {NUM_CUSTOMERS:,}")

## Cell 2 — Helper Functions

In [ ]:
def random_date(start_year=2005, end_year=2023):
    """Generate a random date between start_year and end_year."""
    start = date(start_year, 1, 1)
    end = date(end_year, 12, 31)
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

def weighted_choice(choices, weights):
    """Pick one item from choices according to weights (must sum to 1.0)."""
    r = random.random()
    cumulative = 0.0
    for choice, weight in zip(choices, weights):
        cumulative += weight
        if r < cumulative:
            return choice
    return choices[-1]

# Reference date for 90-day calculations
REFERENCE_DATE = date(2024, 3, 31)
DATE_90_DAYS_AGO = REFERENCE_DATE - timedelta(days=90)

print("✅ Helper functions defined")
print(f"   Reference date: {REFERENCE_DATE}")
print(f"   90-day window start: {DATE_90_DAYS_AGO}")

## Cell 3 — Generate Products Table

The products table is small (10 rows) and defines the product catalogue for Contoso Banque.

In [ ]:
products_data = [
    # (product_id, product_category, product_name, product_family)
    ("PROD-001", "Savings",    "Livret Contoso",          "Retail Banking"),
    ("PROD-002", "Savings",    "Compte Épargne+",          "Retail Banking"),
    ("PROD-003", "Credit",     "Carte Visa Classic",       "Retail Banking"),
    ("PROD-004", "Credit",     "Carte Platinum",           "Retail Banking"),
    ("PROD-005", "Lending",    "Prêt Personnel Contoso",   "Consumer Finance"),
    ("PROD-006", "Lending",    "Crédit Immobilier",        "Consumer Finance"),
    ("PROD-007", "Insurance",  "Assurance Habitation",     "Retail Banking"),
    ("PROD-008", "Insurance",  "Assurance Vie Contoso",    "Wealth Management"),
    ("PROD-009", "Investment", "PEA Contoso",              "Wealth Management"),
    ("PROD-010", "Investment", "Compte-Titres Ordinaire",  "Wealth Management"),
]

products_schema = StructType([
    StructField("product_id",       StringType(), False),
    StructField("product_category", StringType(), True),
    StructField("product_name",     StringType(), True),
    StructField("product_family",   StringType(), True),
])

products_df = spark.createDataFrame(products_data, schema=products_schema)
print(f"✅ Products: {products_df.count()} rows")
products_df.show(truncate=False)

## Cell 4 — Generate Customers Table

Generates 10,000 synthetic customers with demographics and a synthetic churn label.

**Churn label logic:**
- Higher churn probability for: inactive, declining balances, single product, non-digital, age > 70
- Lower churn probability for: multi-product, digital + high activity

In [ ]:
regions = [
    "Île-de-France", "Auvergne-Rhône-Alpes", "Nouvelle-Aquitaine",
    "Occitanie", "Hauts-de-France", "Grand Est",
    "Provence-Alpes-Côte d'Azur", "Pays de la Loire", "Normandie", "Bretagne"
]
region_weights = [0.22, 0.12, 0.10, 0.09, 0.09, 0.08, 0.08, 0.07, 0.08, 0.07]

income_bands  = ["Low", "Medium", "High", "Very High"]
income_weights = [0.25, 0.45, 0.22, 0.08]

risk_profiles  = ["Conservative", "Moderate", "Aggressive"]
risk_weights   = [0.50, 0.35, 0.15]

def compute_churn_probability(
    txn_count_90d, balance_trend, product_count, digital_flag, age
):
    """Synthetic churn probability using transparent business heuristics."""
    prob = 0.10  # Base: 10% churn
    if txn_count_90d == 0:          prob += 0.25  # Inactive
    elif txn_count_90d <= 3:        prob += 0.10  # Low activity
    if balance_trend == "Declining": prob += 0.15
    if product_count == 1:           prob += 0.20
    elif product_count == 0:         prob += 0.25
    if digital_flag == 0:            prob += 0.10
    if age > 70:                     prob += 0.05
    if product_count >= 3:           prob -= 0.15
    if digital_flag == 1 and txn_count_90d >= 10:
                                     prob -= 0.10
    return max(0.02, min(0.95, prob))

customers_data = []
# Pre-generate per-customer transaction and account info for churn labelling
# We'll store these for cross-table consistency
customer_meta = {}  # customer_id -> {txn_count_90d, balance_trend, product_count, digital_flag}

for i in range(1, NUM_CUSTOMERS + 1):
    cid = f"CUST-{i:05d}"
    age = random.randint(18, 85)
    region = weighted_choice(regions, region_weights)
    since_date = random_date(2000, 2022)
    tenure = (REFERENCE_DATE - since_date).days // 30
    income = weighted_choice(income_bands, income_weights)
    risk = weighted_choice(risk_profiles, risk_weights)
    digital = 1 if random.random() < 0.62 else 0

    # Pre-compute synthetic features for churn label consistency
    txn_count_90d = random.choices(
        [0, random.randint(1, 3), random.randint(4, 9), random.randint(10, 30)],
        weights=[0.15, 0.20, 0.35, 0.30]
    )[0]
    balance_trend = weighted_choice(
        ["Growing", "Stable", "Declining"], [0.30, 0.45, 0.25]
    )
    product_count = random.choices(
        [0, 1, 2, 3, 4, 5],
        weights=[0.02, 0.25, 0.32, 0.25, 0.12, 0.04]
    )[0]

    churn_prob = compute_churn_probability(
        txn_count_90d, balance_trend, product_count, digital, age
    )
    churned = 1 if random.random() < churn_prob else 0

    customer_meta[cid] = {
        "txn_count_90d": txn_count_90d,
        "balance_trend": balance_trend,
        "product_count": product_count,
        "digital_flag": digital,
        "churned": churned,
        "since_date": since_date,
    }

    customers_data.append((
        cid, age, region,
        since_date.isoformat(), tenure,
        income, risk, digital, churned
    ))

customers_schema = StructType([
    StructField("customer_id",        StringType(),  False),
    StructField("age",                IntegerType(), True),
    StructField("region",             StringType(),  True),
    StructField("customer_since_date", StringType(), True),
    StructField("tenure_months",      IntegerType(), True),
    StructField("income_band",        StringType(),  True),
    StructField("risk_profile",       StringType(),  True),
    StructField("digital_active_flag", IntegerType(), True),
    StructField("churned_90d",        IntegerType(), True),
])

customers_df = spark.createDataFrame(customers_data, schema=customers_schema)

total = customers_df.count()
churned_count = customers_df.filter(col("churned_90d") == 1).count()
churn_rate = 100.0 * churned_count / total

print(f"✅ Customers: {total:,} rows")
print(f"   Churned: {churned_count:,} ({churn_rate:.1f}%)")
print(f"   Retained: {total - churned_count:,} ({100 - churn_rate:.1f}%)")
customers_df.show(5)

## Cell 5 — Generate Accounts Table

Generates ~15,000 bank accounts. Each customer has 1–3 accounts. Balances and trends are consistent with the churn label generated for each customer.

In [ ]:
account_types = ["Checking", "Savings", "Joint", "Business"]
account_type_weights = [0.55, 0.25, 0.12, 0.08]

accounts_data = []
account_meta = {}  # account_id -> customer_id (for transactions)

acc_counter = 1
for cid, meta in customer_meta.items():
    num_accounts = random.choices([1, 2, 3], weights=[0.55, 0.35, 0.10])[0]

    for _ in range(num_accounts):
        aid = f"ACC-{acc_counter:06d}"
        acc_counter += 1

        acct_type = weighted_choice(account_types, account_type_weights)
        open_date = meta["since_date"] + timedelta(days=random.randint(0, 180))

        # Balance influenced by balance_trend from customer meta
        if meta["balance_trend"] == "Growing":
            base = random.uniform(5000, 50000)
            avg_90 = base * random.uniform(0.75, 0.95)
        elif meta["balance_trend"] == "Declining":
            base = random.uniform(100, 15000)
            avg_90 = base * random.uniform(1.05, 1.25)
        else:  # Stable
            base = random.uniform(1000, 30000)
            avg_90 = base * random.uniform(0.92, 1.08)

        # Churned customers tend to have lower balances
        if meta["churned"] == 1:
            base = min(base, 8000)
            avg_90 = min(avg_90, 8000)

        account_meta[aid] = cid

        accounts_data.append((
            aid,
            cid,
            acct_type,
            open_date.isoformat(),
            round(base, 2),
            round(avg_90, 2),
            meta["balance_trend"],
        ))

accounts_schema = StructType([
    StructField("account_id",     StringType(),  False),
    StructField("customer_id",    StringType(),  True),
    StructField("account_type",   StringType(),  True),
    StructField("open_date",      StringType(),  True),
    StructField("current_balance", DoubleType(), True),
    StructField("avg_balance_90d", DoubleType(), True),
    StructField("balance_trend",  StringType(),  True),
])

accounts_df = spark.createDataFrame(accounts_data, schema=accounts_schema)
print(f"✅ Accounts: {accounts_df.count():,} rows")
accounts_df.show(5)

## Cell 6 — Generate Customer Products Table

Assigns products to customers based on the `product_count` from customer_meta. This ensures consistency with the churn label (single-product customers have higher churn).

In [ ]:
product_ids = [p[0] for p in products_data]  # PROD-001 through PROD-010

customer_products_data = []

for cid, meta in customer_meta.items():
    n_products = meta["product_count"]
    if n_products == 0:
        continue

    assigned = random.sample(product_ids, min(n_products, len(product_ids)))

    for pid in assigned:
        start_date = meta["since_date"] + timedelta(days=random.randint(0, 365))
        # Churned customers may have cancelled some products
        if meta["churned"] == 1 and random.random() < 0.20:
            active = 0
        else:
            active = 1

        customer_products_data.append((
            cid,
            pid,
            start_date.isoformat(),
            active,
        ))

cp_schema = StructType([
    StructField("customer_id",         StringType(),  False),
    StructField("product_id",          StringType(),  True),
    StructField("product_start_date",  StringType(),  True),
    StructField("active_product_flag", IntegerType(), True),
])

customer_products_df = spark.createDataFrame(customer_products_data, schema=cp_schema)
print(f"✅ Customer Products: {customer_products_df.count():,} rows")
customer_products_df.show(5)

## Cell 7 — Generate Transactions Table

Generates ~175,000 transactions. Transaction volume is directly tied to `txn_count_90d` from customer_meta, ensuring consistency with the churn label.

In [ ]:
txn_types    = ["Debit", "Credit", "Transfer", "Direct Debit"]
txn_weights  = [0.45, 0.30, 0.15, 0.10]

channels      = ["Online", "Mobile", "ATM", "Branch", "POS"]
channel_weights = [0.30, 0.30, 0.15, 0.10, 0.15]

# Non-digital customers use branch/ATM more
channel_weights_nond = [0.10, 0.05, 0.35, 0.35, 0.15]

merchant_cats = [
    "Grocery", "Restaurant", "Travel", "Utilities",
    "Healthcare", "Retail", "Entertainment", "Education",
    "Subscription", "Transport", "Insurance", "Other"
]
mc_weights = [0.18, 0.12, 0.08, 0.10, 0.06, 0.14, 0.07, 0.04, 0.08, 0.09, 0.05, 0.09]

transactions_data = []
txn_counter = 1

account_ids_list = list(account_meta.keys())

# Build account_id -> customer_id map
acc_to_cust = account_meta  # {account_id: customer_id}

# Build customer_id -> list of account_ids
cust_to_accounts = {}
for aid, cid in acc_to_cust.items():
    cust_to_accounts.setdefault(cid, []).append(aid)

for cid, meta in customer_meta.items():
    cust_accounts = cust_to_accounts.get(cid, [])
    if not cust_accounts:
        continue

    digital = meta["digital_flag"]

    # Generate last-90-day transactions (used for churn labelling)
    n_recent = meta["txn_count_90d"]
    # Generate historical transactions (last 12 months, outside 90-day window)
    n_hist = random.randint(2, 20)

    for _ in range(n_recent):
        aid = random.choice(cust_accounts)
        txn_date = DATE_90_DAYS_AGO + timedelta(days=random.randint(0, 89))
        txn_type = weighted_choice(txn_types, txn_weights)
        ch_w = channel_weights if digital else channel_weights_nond
        channel = weighted_choice(channels, ch_w)
        amount = round(random.uniform(5, 2000), 2)
        if txn_type in ["Debit", "Direct Debit"]:
            amount = -amount
        mc = weighted_choice(merchant_cats, mc_weights)
        transactions_data.append((
            f"TXN-{txn_counter:08d}",
            aid, cid,
            txn_date.isoformat(),
            txn_type, channel,
            amount, mc
        ))
        txn_counter += 1

    for _ in range(n_hist):
        aid = random.choice(cust_accounts)
        days_back = random.randint(91, 365)
        txn_date = REFERENCE_DATE - timedelta(days=days_back)
        txn_type = weighted_choice(txn_types, txn_weights)
        ch_w = channel_weights if digital else channel_weights_nond
        channel = weighted_choice(channels, ch_w)
        amount = round(random.uniform(5, 2000), 2)
        if txn_type in ["Debit", "Direct Debit"]:
            amount = -amount
        mc = weighted_choice(merchant_cats, mc_weights)
        transactions_data.append((
            f"TXN-{txn_counter:08d}",
            aid, cid,
            txn_date.isoformat(),
            txn_type, channel,
            amount, mc
        ))
        txn_counter += 1

txn_schema = StructType([
    StructField("transaction_id",   StringType(),  False),
    StructField("account_id",       StringType(),  True),
    StructField("customer_id",      StringType(),  True),
    StructField("transaction_date", StringType(),  True),
    StructField("transaction_type", StringType(),  True),
    StructField("channel",          StringType(),  True),
    StructField("amount",           DoubleType(),  True),
    StructField("merchant_category", StringType(), True),
])

transactions_df = spark.createDataFrame(transactions_data, schema=txn_schema)
print(f"✅ Transactions: {transactions_df.count():,} rows")
transactions_df.show(5)

## Cell 8 — Write Raw Data to Files/churn/raw/

Write raw datasets to the `Files/churn/raw/` section of the Lakehouse in Parquet format.
This simulates a raw ingestion landing zone.

In [ ]:
RAW_PATH = "Files/churn/raw"

print("Writing raw data to Files/churn/raw/ ...")

customers_df.write.mode("overwrite").parquet(f"{RAW_PATH}/customers")
print(f"  ✅ customers → {RAW_PATH}/customers")

accounts_df.write.mode("overwrite").parquet(f"{RAW_PATH}/accounts")
print(f"  ✅ accounts → {RAW_PATH}/accounts")

products_df.write.mode("overwrite").parquet(f"{RAW_PATH}/products")
print(f"  ✅ products → {RAW_PATH}/products")

customer_products_df.write.mode("overwrite").parquet(f"{RAW_PATH}/customer_products")
print(f"  ✅ customer_products → {RAW_PATH}/customer_products")

transactions_df.write.mode("overwrite").parquet(f"{RAW_PATH}/transactions")
print(f"  ✅ transactions → {RAW_PATH}/transactions")

print("\n✅ All raw files written successfully.")

## Cell 9 — Write Delta Tables to Tables/

Write the same datasets as Delta tables under `Tables/`. These are the tables that will be:
- Queryable via the SQL analytics endpoint
- Available in Power BI semantic models
- Connected to the Fabric Data Agent

In [ ]:
print("Writing Delta tables to Tables/ ...")

customers_df.write.format("delta").mode("overwrite").saveAsTable("customers")
print("  ✅ Delta table: customers")

accounts_df.write.format("delta").mode("overwrite").saveAsTable("accounts")
print("  ✅ Delta table: accounts")

products_df.write.format("delta").mode("overwrite").saveAsTable("products")
print("  ✅ Delta table: products")

customer_products_df.write.format("delta").mode("overwrite").saveAsTable("customer_products")
print("  ✅ Delta table: customer_products")

transactions_df.write.format("delta").mode("overwrite").saveAsTable("transactions")
print("  ✅ Delta table: transactions")

print("\n✅ All Delta tables written successfully.")

## Cell 10 — Validation

Verify all tables were created correctly.

In [ ]:
print("=" * 55)
print("VALIDATION — Table Row Counts")
print("=" * 55)

validation_results = [
    ("customers",         spark.sql("SELECT COUNT(*) AS n FROM customers").collect()[0][0]),
    ("accounts",          spark.sql("SELECT COUNT(*) AS n FROM accounts").collect()[0][0]),
    ("products",          spark.sql("SELECT COUNT(*) AS n FROM products").collect()[0][0]),
    ("customer_products", spark.sql("SELECT COUNT(*) AS n FROM customer_products").collect()[0][0]),
    ("transactions",      spark.sql("SELECT COUNT(*) AS n FROM transactions").collect()[0][0]),
]

for table, count in validation_results:
    print(f"  {table:<25} {count:>10,} rows")

print()
print("Churn rate summary:")
spark.sql("""
    SELECT
        COUNT(*) AS total_customers,
        SUM(churned_90d) AS churned,
        ROUND(100.0 * SUM(churned_90d) / COUNT(*), 2) AS churn_rate_pct
    FROM customers
""").show()

print("✅ Notebook 01 complete. Proceed to Notebook 02.")